In [ ]:
combo = ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID']

In [ ]:
import pyspark.sql.functions as F


def get_non_unique_rows(df, combo, limit_display=20):
    """
    Retourne et affiche les lignes qui partagent des valeurs dupliquées 
    pour une combinaison de colonnes spécifique.
    
    :param df: Le DataFrame Spark à analyser
    :param combo: Liste des colonnes de la combinaison (ex: ['id_magasin', 'num_ticket'])
    :param limit_display: Nombre maximal de lignes dupliquées à afficher dans la console
    """
    print(f"🔍 Recherche des doublons pour la combinaison : {combo}...\n")
    
    # 1. Définir une fenêtre Spark sur la combinaison pour compter les occurrences
    window_spec = Window.partitionBy(*[F.col(c) for c in combo])
    
    # 2. Ajouter une colonne temporaire avec le nombre de répétitions
    df_with_counts = df.withColumn("_count_occurrence", F.count("*").over(window_spec))
    
    # 3. Filtrer pour ne garder que les lignes présentes plus d'une fois
    duplicates_df = df_with_counts.filter(F.col("_count_occurrence") > 1)
    
    # 4. Trier par les colonnes de la combinaison pour grouper visuellement les doublons ensemble
    duplicates_df = duplicates_df.sort(*combo, "_count_occurrence", ascending=False)
    
    # Calculer le nombre total de lignes en anomalie
    total_duplicate_rows = duplicates_df.count()
    
    if total_duplicate_rows == 0:
        print(f"✅ Félicitations ! La combinaison {combo} est parfaitement unique. Aucune ligne dupliquée.")
        return df.sparkSession.createDataFrame([], df.schema)
    
    # Trouver le nombre de "clés" distinctes dupliquées
    distinct_duplicate_keys = duplicates_df.select(combo).distinct().count()
    
    print(f"⚠️ Alerte Doublons :")
    print(f"   • {total_duplicate_rows:,} lignes totales sont impliquées dans des doublons.")
    print(f"   • Il y a {distinct_duplicate_keys:,} valeur(s) de combinaison distincte(s) qui se répètent.")
    print(f"\nAffichage des {min(limit_display, total_duplicate_rows)} premières lignes dupliquées (ordonnées par groupe) :")
    
    # Afficher le résultat (en incluant la colonne de compteur à la fin pour voir le nombre exact de répétitions)
    duplicates_df.show(total_duplicate_rows, truncate=False)
    
    # Retourne le DataFrame contenant uniquement les lignes dupliquées (sans la colonne technique de décompte)
    return duplicates_df.drop("_count_occurrence")

In [ ]:
duplicates_df= get_non_unique_rows

In [ ]:

from datetime import datetime

current_timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M")

spark.sql("CREATE SCHEMA IF NOT EXISTS nyc_taxi.quranatine")
duplicates.write.mode("overwrite").saveAsTable(f"nyc_taxi.quranatine.taxi_trips_duplicates_{current_timestamp}_for_{"_".join(combo)}")